# 自由现金流（FCF）模型实战教程

**目标**：用模拟财务数据，走通"自由现金流计算 → 因子构建 → DCF估值 → 回测验证"的完整流水线。

**特色**：完全自包含，使用模拟数据，无需外部 API 依赖，立即可运行。

**依赖**：仅需要 `pandas numpy matplotlib`

## 1. 自由现金流的核心概念

### 什么是自由现金流

> 自由现金流 = 企业经营产生的现金 - 维持经营必需的现金支出

**简单理解**：这是企业"真正自由"可用的现金，可以：
- 分红给股东
- 回购股票
- 投资新项目
- 偿还债务

### 为什么自由现金流重要

- **利润可能造假**，但现金流很难造假
- **现金流驱动的增长**比利润驱动更健康
- **高自由现金流企业**通常是优质公司（护城河强）
- **自由现金流因子**在量化投资中有显著超额收益

### 基础公式

```
FCF = 经营活动现金流 - 资本支出
```

- **经营活动现金流**：企业日常经营产生的现金（卖产品收到的钱 - 买原材料付的钱）
- **资本支出**：购买设备、建厂房等固定资产投资

## 2. 环境设置

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 中文字体设置
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪，仅依赖 pandas + numpy + matplotlib')

## 3. 创建模拟财务数据

为演示目的，我们创建一家稳定增长消费品企业的模拟财务数据：
- 5 年历史数据（2020-2024）
- 3 年预测数据（2025-2027）
- 包含三大报表：现金流量表、利润表、资产负债表

In [ ]:
def create_mock_company_financials(company_name: str = '模拟消费品公司') -> dict:
    """创建完整的模拟财务数据
    
    Args:
        company_name: 公司名称
    
    Returns:
        包含三大报表的字典
    """
    years = list(range(2020, 2028))  # 2020-2027 (8年)
    
    # 基础参数（2020年）
    base_revenue = 100.0  # 亿元
    net_profit_margin = 0.15
    revenue_growth_rates = [0.08, 0.10, 0.09, 0.07, 0.08,  # 历史增长率
                           0.08, 0.07, 0.06]  # 预测增长率
    
    data = []
    current_revenue = base_revenue
    
    for i, year in enumerate(years):
        # 营业收入（历史有波动，预测平滑）
        if i < 5:
            volatility = np.random.uniform(-0.02, 0.02)
            current_revenue *= (1 + revenue_growth_rates[i] + volatility)
        else:
            current_revenue *= (1 + revenue_growth_rates[i])
        
        # 计算其他财务指标
        net_profit = current_revenue * net_profit_margin
        
        # 现金流量表数据
        operating_cf = net_profit * 0.95  # 经营现金流略高于净利润
        capex = current_revenue * 0.08  # 资本支出占营收的8%
        fcf = operating_cf - capex
        
        # 资产负债表数据
        total_assets = current_revenue * 1.5
        net_assets = total_assets * 0.6
        
        # 数据类型标记
        data_type = '历史' if i < 5 else '预测'
        
        data.append({
            '年份': year,
            '数据类型': data_type,
            # 利润表
            '营业收入(亿)': round(current_revenue, 2),
            '净利润(亿)': round(net_profit, 2),
            # 现金流量表
            '经营活动现金流(亿)': round(operating_cf, 2),
            '资本支出(亿)': round(capex, 2),
            '自由现金流(亿)': round(fcf, 2),
            # 资产负债表
            '总资产(亿)': round(total_assets, 2),
            '净资产(亿)': round(net_assets, 2),
        })
    
    financials = pd.DataFrame(data)
    
    # 分离为三个报表
    cash_flow = financials[['年份', '数据类型', '经营活动现金流(亿)', '资本支出(亿)', '自由现金流(亿)']]
    profit = financials[['年份', '数据类型', '营业收入(亿)', '净利润(亿)']]
    balance = financials[['年份', '数据类型', '总资产(亿)', '净资产(亿)']]
    
    return {
        'name': company_name,
        'cash_flow': cash_flow,
        'profit': profit, 
        'balance': balance,
        'full_data': financials
    }

# 创建模拟公司数据
mock_company = create_mock_company_financials('优质消费品公司')
print(f"模拟公司：{mock_company['name']}")
print(f"数据期间：{mock_company['cash_flow']['年份'].min()} - {mock_company['cash_flow']['年份'].max()}")
print(f"总期数：{len(mock_company['full_data'])} 期")

# 显示完整财务数据
mock_company['full_data']

## 4. 计算自由现金流

**基础公式**：`FCF = 经营活动现金流 - 资本支出`

**更精确的公式**：`FCF = 经营活动现金流 - 资本支出 + 利息收入 - 利息支出`

In [ ]:
def calculate_fcf(cash_flow_df: pd.DataFrame) -> pd.DataFrame:
    """计算自由现金流
    
    Args:
        cash_flow_df: 现金流量表 DataFrame
    
    Returns:
        包含 FCF 的 DataFrame
    """
    try:
        # 获取关键数据
        operating_cf = cash_flow_df['经营活动现金流(亿)']
        capex = cash_flow_df['资本支出(亿)']
        
        # 计算 FCF
        fcf = operating_cf - capex
        
        result_df = cash_flow_df.copy()
        result_df['自由现金流(亿)'] = fcf
        
        return result_df
        
    except Exception as e:
        print(f"计算 FCF 失败: {e}")
        return pd.DataFrame()

# 计算 FCF
fcf_df = calculate_fcf(mock_company['cash_flow'])
print("FCF 计算成功！")
print(f"最近一期 FCF: {fcf_df['自由现金流(亿)'].iloc[-1]:,.2f} 亿元")
fcf_df.tail(3)

## 5. 构建 FCF 相关因子

### 常见 FCF 因子

| 因子名 | 公式 | 含义 |
|-------|------|------|
| **FCF/营收** | 自由现金流 / 营业收入 | 每元营收产生的自由现金流 |
| **FCF/净利润** | 自由现金流 / 净利润 | 现金流对利润的覆盖程度 |
| **FCF/总资产** | 自由现金流 / 总资产 | 资产产生现金流的能力 |
| **FCF 增长率** | (本期 FCF - 上期 FCF) / |上期 FCF| | FCF 增长趋势 |
| **FCF 稳定性** | FCF 标准差 / |FCF 均值| | FCF 波动程度 |
| **FCF/市值** | 自由现金流 / 总市值 | 现金流估值水平 |

In [ ]:
def build_fcf_factors(cash_flow_df: pd.DataFrame, profit_df: pd.DataFrame, 
                       balance_df: pd.DataFrame, market_cap: float = None) -> pd.DataFrame:
    """构建 FCF 相关因子
    
    Args:
        cash_flow_df: 现金流量表
        profit_df: 利润表
        balance_df: 资产负债表
        market_cap: 总市值（可选）
    
    Returns:
        包含 FCF 因子的 DataFrame
    """
    if cash_flow_df.empty:
        return pd.DataFrame()
    
    factors = pd.DataFrame(index=cash_flow_df.index)
    factors['自由现金流(亿)'] = cash_flow_df['自由现金流(亿)']
    
    # FCF/营收比
    if '营业收入(亿)' in profit_df.columns:
        factors['FCF_营收比'] = factors['自由现金流(亿)'] / (profit_df['营业收入(亿)'].abs() + 1e-6)
    
    # FCF/净利润比
    if '净利润(亿)' in profit_df.columns:
        factors['FCF_净利润比'] = factors['自由现金流(亿)'] / (profit_df['净利润(亿)'].abs() + 1e-6)
    
    # FCF/总资产比
    if '总资产(亿)' in balance_df.columns:
        factors['FCF_总资产比'] = factors['自由现金流(亿)'] / (balance_df['总资产(亿)'].abs() + 1e-6)
    
    # FCF 增长率
    factors['FCF_增长率'] = factors['自由现金流(亿)'].pct_change()
    
    # FCF 稳定性（最近 4 期的标准差/均值）
    if len(factors) >= 4:
        recent_fcf = factors['自由现金流(亿)'].tail(4)
        factors['FCF_稳定性'] = recent_fcf.std() / (recent_fcf.abs().mean() + 1e-6)
    
    # FCF/市值
    if market_cap is not None:
        factors['FCF_市值比'] = factors['自由现金流(亿)'] / market_cap
    
    return factors

# 构建因子（假设总市值 500 亿元）
factors = build_fcf_factors(fcf_df, mock_company['profit'], mock_company['balance'], market_cap=500.0)
print("因子构建成功！")
factors.tail()

## 6. 可视化 FCF 趋势

In [ ]:
def plot_fcf_trend(factors_df: pd.DataFrame, company_name: str):
    """绘制 FCF 趋势图
    
    Args:
        factors_df: 因子 DataFrame
        company_name: 公司名称
    """
    if factors_df.empty:
        print("无数据可绘制")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. FCF 绝对值趋势
    axes[0, 0].plot(range(len(factors_df)), factors_df['自由现金流(亿)'], 
                    marker='o', linewidth=2, color='steelblue')
    axes[0, 0].set_title(f'{company_name} - 自由现金流趋势')
    axes[0, 0].set_ylabel('金额（亿元）')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. FCF/营收比
    if 'FCF_营收比' in factors_df.columns:
        axes[0, 1].bar(range(len(factors_df)), factors_df['FCF_营收比'], 
                       color='coral', alpha=0.7)
        axes[0, 1].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
        axes[0, 1].set_title('FCF/营收比')
        axes[0, 1].set_ylabel('比率')
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. FCF 增长率
    if 'FCF_增长率' in factors_df.columns:
        axes[1, 0].bar(range(len(factors_df)), factors_df['FCF_增长率'], 
                       color=['green' if x > 0 else 'red' for x in factors_df['FCF_增长率']], 
                       alpha=0.7)
        axes[1, 0].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
        axes[1, 0].set_title('FCF 增长率')
        axes[1, 0].set_ylabel('增长率')
        axes[1, 0].grid(True, alpha=0.3)
    
    # 4. 综合指标雷达图
    if all(col in factors_df.columns for col in ['FCF_营收比', 'FCF_净利润比', 'FCF_总资产比']):
        categories = ['FCF/营收', 'FCF/净利润', 'FCF/总资产']
        values = [factors_df[cat].iloc[-1] for cat in ['FCF_营收比', 'FCF_净利润比', 'FCF_总资产比']]
        
        # 归一化到 0-1 范围
        values = [min(max(v, 0), 1) for v in values]
        
        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
        values += values[:1]  # 闭合
        angles += angles[:1]  # 闭合
        
        ax_radar = fig.add_subplot(2, 2, 4, polar=True)
        ax_radar.plot(angles, values, 'o-', linewidth=2, color='purple')
        ax_radar.fill(angles, values, alpha=0.25, color='purple')
        ax_radar.set_xticks(angles[:-1])
        ax_radar.set_xticklabels(categories)
        ax_radar.set_ylim(0, 1)
        ax_radar.set_title('FCF 综合指标')
    
    plt.tight_layout()
    plt.show()

# 绘制图表
plot_fcf_trend(factors, mock_company['name'])

## 7. 多公司对比分析

创建多个模拟公司进行横向对比，识别优质标的。

In [ ]:
def create_multiple_companies(count: int = 5) -> list:
    """创建多个模拟公司
    
    Args:
        count: 公司数量
    
    Returns:
        公司列表
    """
    companies = []
    company_types = [
        '优质消费品', '稳定制造业', '成长科技', '周期性企业', '传统金融'
    ]
    
    for i in range(count):
        name = f"{company_types[i % len(company_types)]}公司{i+1}"
        # 为不同公司设置不同特征
        np.random.seed(i * 42)  # 确保可重复性
        companies.append(create_mock_company_financials(name))
    
    return companies

def compare_fcf_across_companies(companies: list) -> pd.DataFrame:
    """横向对比不同公司的 FCF 表现
    
    Args:
        companies: 公司列表
    
    Returns:
        对比结果 DataFrame
    """
    comparison = []
    
    for company in companies:
        try:
            fcf_df = calculate_fcf(company['cash_flow'])
            if fcf_df.empty:
                continue
            
            # 计算关键指标
            latest_fcf = fcf_df['自由现金流(亿)'].iloc[-1]
            fcf_trend = fcf_df['自由现金流(亿)'].tail(3).mean()  # 最近3期平均
            fcf_growth = fcf_df['自由现金流(亿)'].pct_change().tail(3).mean()  # 最近3期平均增长率
            
            comparison.append({
                '公司名称': company['name'],
                '最新FCF(亿)': latest_fcf,
                '平均FCF(亿)': fcf_trend,
                'FCF增长率': fcf_growth,
                '数据期数': len(fcf_df)
            })
        except Exception as e:
            print(f"处理 {company['name']} 失败: {e}")
            continue
    
    if not comparison:
        return pd.DataFrame()
    
    return pd.DataFrame(comparison).sort_values('最新FCF(亿)', ascending=False)

# 创建多个公司并进行对比
companies = create_multiple_companies(5)
comparison_df = compare_fcf_across_companies(companies)
print("多公司 FCF 对比结果：")
comparison_df

In [ ]:
def plot_fcf_comparison(comparison_df: pd.DataFrame):
    """绘制 FCF 对比图
    
    Args:
        comparison_df: 对比结果 DataFrame
    """
    if comparison_df.empty:
        print("无数据可绘制")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. 最新 FCF 对比
    colors = plt.cm.Set3(range(len(comparison_df)))
    axes[0].barh(comparison_df['公司名称'], comparison_df['最新FCF(亿)'], 
                 color=colors, alpha=0.8)
    axes[0].set_xlabel('最新 FCF（亿元）')
    axes[0].set_title('最新自由现金流对比')
    axes[0].grid(True, alpha=0.3, axis='x')
    
    # 2. FCF 增长率对比
    growth_colors = ['green' if x > 0 else 'red' for x in comparison_df['FCF增长率']]
    axes[1].barh(comparison_df['公司名称'], comparison_df['FCF增长率'] * 100, 
                 color=growth_colors, alpha=0.7)
    axes[1].axvline(x=0, color='black', linestyle='--', linewidth=0.8)
    axes[1].set_xlabel('FCF 增长率 (%)')
    axes[1].set_title('FCF 增长趋势对比')
    axes[1].grid(True, alpha=0.3, axis='x')
    
    # 3. 散点图：平均 FCF vs 增长率
    axes[2].scatter(comparison_df['平均FCF(亿)'], comparison_df['FCF增长率'] * 100, 
                    s=200, alpha=0.7, color=colors)
    
    # 标注公司名称
    for i, row in comparison_df.iterrows():
        axes[2].annotate(row['公司名称'], 
                        (row['平均FCF(亿)'], row['FCF增长率'] * 100),
                        fontsize=9, alpha=0.8)
    
    axes[2].set_xlabel('平均 FCF（亿元）')
    axes[2].set_ylabel('FCF 增长率 (%)')
    axes[2].set_title('FCF 质量（规模 vs 增长）')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 绘制对比图
plot_fcf_comparison(comparison_df)

## 8. FCF 选股策略框架

### 基础选股条件

```
1. FCF > 0（正自由现金流）
2. FCF 增长率 > 10%（现金流持续增长）
3. FCF/营收比 > 10%（现金流质量高）
4. FCF/净利润比 > 80%（利润现金流覆盖）
```

In [ ]:
def fcf_screening(company_data: dict, market_cap: float = 500.0) -> list:
    """FCF 选股筛选
    
    Args:
        company_data: 公司财务数据
        market_cap: 总市值（亿元）
    
    Returns:
        筛选结果
    """
    try:
        fcf_df = calculate_fcf(company_data['cash_flow'])
        profit_df = company_data['profit']
        balance_df = company_data['balance']
        
        if fcf_df.empty or len(fcf_df) < 3:
            return ['数据不足']
        
        # 构建因子
        factors = build_fcf_factors(fcf_df, profit_df, balance_df, market_cap)
        if factors.empty:
            return ['因子构建失败']
        
        # 最新数据
        latest = factors.iloc[-1]
        
        # 筛选条件
        results = []
        
        # 1. 正自由现金流
        if latest['自由现金流(亿)'] > 0:
            results.append("✅ 正现金流")
        else:
            results.append("❌ 负现金流")
        
        # 2. FCF 增长率
        if 'FCF_增长率' in latest and not pd.isna(latest['FCF_增长率']):
            if latest['FCF_增长率'] > 0.1:  # 10%
                results.append("✅ 增长率高")
            else:
                results.append("⚠️ 增长率一般")
        
        # 3. FCF/营收比
        if 'FCF_营收比' in latest and not pd.isna(latest['FCF_营收比']):
            if latest['FCF_营收比'] > 0.1:  # 10%
                results.append("✅ 现金流质量高")
            else:
                results.append("⚠️ 现金流质量一般")
        
        # 4. FCF/净利润比
        if 'FCF_净利润比' in latest and not pd.isna(latest['FCF_净利润比']):
            if latest['FCF_净利润比'] > 0.8:  # 80%
                results.append("✅ 利润现金流覆盖好")
            else:
                results.append("⚠️ 利润现金流覆盖一般")
        
        return results
        
    except Exception as e:
        return [f"分析失败: {e}"]

# 测试筛选所有公司
print("多公司 FCF 选股结果：")
for company in companies:
    screening_results = fcf_screening(company, market_cap=500.0)
    print(f"\n{company['name']}：")
    for result in screening_results:
        print(f"  {result}")

## 9. DCF 估值模型（实战）

### DCF (Discounted Cash Flow) 估值原理

> 企业价值 = 未来自由现金流的现值总和

**核心公式**：
```
企业价值 = ∑(未来FCF / (1+折现率)^t) + 终值
股权价值 = 企业价值 - 净债务
每股价值 = 股权价值 / 股本总数
```

### 9.1 三阶段 DCF 模型

```
1. 预测期价值 (通常3-5年)：未来FCF的现值总和
2. 终值 (永续增长期)：预测期后按固定增长率永续增长
3. 企业价值 = 预测期价值 + 终值
```

In [ ]:
def dcf_valuation(cash_flow_df: pd.DataFrame, 
                current_fcf: float, 
                forecast_years: int = 3,
                discount_rate: float = 0.10,
                terminal_growth_rate: float = 0.03,
                net_debt: float = 50.0,
                total_shares: float = 10.0) -> dict:
    """DCF 估值模型
    
    Args:
        cash_flow_df: 历史FCF数据
        current_fcf: 当前年度FCF (亿元)
        forecast_years: 预测年数
        discount_rate: 折现率 (WACC)
        terminal_growth_rate: 终期增长率 (通常2-3%)
        net_debt: 净债务 (亿元)
        total_shares: 总股本 (亿股)
    
    Returns:
        估值结果字典
    """
    
    # 计算历史增长率
    if len(cash_flow_df) >= 3:
        historical_fcf = cash_flow_df['自由现金流(亿)'].values
        # 计算平均增长率
        growth_rates = []
        for i in range(1, len(historical_fcf)):
            if historical_fcf[i-1] > 0:
                growth_rates.append((historical_fcf[i] - historical_fcf[i-1]) / historical_fcf[i-1])
        
        if growth_rates:
            avg_growth_rate = np.mean(growth_rates)
            # 保守起见，取历史增长率的一半
            base_growth_rate = max(avg_growth_rate * 0.5, terminal_growth_rate)
        else:
            base_growth_rate = terminal_growth_rate + 0.02
    else:
        base_growth_rate = terminal_growth_rate + 0.02
    
    # 预测期现金流（增长率逐年递减）
    forecasted_fcf = []
    growth_schedule = [base_growth_rate, base_growth_rate * 0.8, base_growth_rate * 0.6]
    
    forecast_fcf = current_fcf
    for i in range(forecast_years):
        growth_rate = growth_schedule[i] if i < len(growth_schedule) else terminal_growth_rate
        forecast_fcf *= (1 + growth_rate)
        forecasted_fcf.append(forecast_fcf)
    
    # 计算预测期现值
    pv_forecast = []
    for i, fcf in enumerate(forecasted_fcf):
        pv = fcf / ((1 + discount_rate) ** (i + 1))
        pv_forecast.append(pv)
    
    pv_forecast_total = sum(pv_forecast)
    
    # 计算终值
    terminal_fcf = forecasted_fcf[-1] * (1 + terminal_growth_rate)
    terminal_value = terminal_fcf / (discount_rate - terminal_growth_rate)
    pv_terminal = terminal_value / ((1 + discount_rate) ** forecast_years)
    
    # 企业价值
    enterprise_value = pv_forecast_total + pv_terminal
    
    # 股权价值
    equity_value = enterprise_value - net_debt
    
    # 每股价值
    share_value = equity_value / total_shares
    
    return {
        '预测期现金流(亿)': forecasted_fcf,
        '预测期现值(亿)': pv_forecast,
        '预测期现值总计(亿)': round(pv_forecast_total, 2),
        '终期现金流(亿)': round(terminal_fcf, 2),
        '终值(亿)': round(terminal_value, 2),
        '终值现值(亿)': round(pv_terminal, 2),
        '企业价值(亿)': round(enterprise_value, 2),
        '净债务(亿)': net_debt,
        '股权价值(亿)': round(equity_value, 2),
        '总股本(亿股)': total_shares,
        '每股价值(元)': round(share_value, 2),
        '折现率': discount_rate,
        '终期增长率': terminal_growth_rate,
        '历史平均增长率': round(avg_growth_rate if growth_rates else 0, 3)
    }

# 对多个公司进行 DCF 估值
valuation_results = []
for company in companies:
    historical_data = company['cash_flow'][company['cash_flow']['数据类型'] == '历史'].tail(3)
    current_fcf = company['cash_flow'][company['cash_flow']['数据类型'] == '历史']['自由现金流(亿)'].iloc[-1]
    
    result = dcf_valuation(
        historical_data, 
        current_fcf,
        forecast_years=3,
        discount_rate=0.10,
        terminal_growth_rate=0.03,
        net_debt=50.0,
        total_shares=10.0
    )
    result['公司名称'] = company['name']
    valuation_results.append(result)

# 显示估值结果
valuation_df = pd.DataFrame(valuation_results)
print("多公司 DCF 估值结果：")
print(valuation_df[['公司名称', '每股价值(元)', '企业价值(亿)', '股权价值(亿)']].round(2))

### 9.2 DCF 估值可视化分析

In [ ]:
def plot_dcf_breakdown(valuation_result: dict, current_price: float = 25.0):
    """绘制DCF估值分解图
    
    Args:
        valuation_result: DCF估值结果
        current_price: 当前股价
    """
    fig = plt.figure(figsize=(15, 8))
    
    # 1. 现金流分解图
    ax1 = plt.subplot(2, 3, 1)
    years = [f'第{i+1}年' for i in range(len(valuation_result['预测期现金流(亿)']))]
    colors = plt.cm.Blues(range(len(years)))
    
    bars = ax1.bar(years, valuation_result['预测期现金流(亿)'], color=colors, alpha=0.7)
    ax1.set_ylabel('FCF (亿元)')
    ax1.set_title('预测期现金流')
    ax1.grid(True, alpha=0.3)
    
    # 添加数值标签
    for bar, val in zip(bars, valuation_result['预测期现金流(亿)']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)
    
    # 2. 企业价值分解饼图
    ax2 = plt.subplot(2, 3, 2)
    value_components = [
        valuation_result['预测期现值总计(亿)'],
        valuation_result['终值现值(亿)']
    ]
    labels = [f'预测期现值\n{value_components[0]:.1f}亿', 
              f'终值现值\n{value_components[1]:.1f}亿']
    colors = ['lightblue', 'lightcoral']
    
    wedges, texts, autotexts = ax2.pie(value_components, labels=labels, colors=colors, 
                                        autopct='%1.1f%%', startangle=90)
    ax2.set_title('企业价值构成')
    
    # 3. 敏感性分析
    ax3 = plt.subplot(2, 3, 3)
    discount_rates = np.linspace(0.08, 0.12, 9)
    share_values = []
    
    for dr in discount_rates:
        result = dcf_valuation(historical_data, current_fcf, 
                              discount_rate=dr, terminal_growth_rate=0.03,
                              net_debt=50.0, total_shares=10.0)
        share_values.append(result['每股价值(元)'])
    
    ax3.plot(discount_rates * 100, share_values, 'b-', linewidth=2, marker='o')
    ax3.axhline(y=current_price, color='r', linestyle='--', label=f'当前价 {current_price:.1f}元')
    ax3.set_xlabel('折现率 (%)')
    ax3.set_ylabel('每股价值 (元)')
    ax3.set_title('折现率敏感性分析')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. 估值总结表
    ax4 = plt.subplot(2, 3, 4)
    ax4.axis('off')
    
    safety_margin = (valuation_result['每股价值(元)'] - current_price) / current_price * 100
    
    summary_data = [
        ['项目', '数值'],
        ['企业价值', f"{valuation_result['企业价值(亿)']:.1f} 亿元"],
        ['股权价值', f"{valuation_result['股权价值(亿)']:.1f} 亿元"],
        ['每股价值', f"{valuation_result['每股价值(元)']:.2f} 元"],
        ['当前股价', f"{current_price:.2f} 元"],
        ['安全边际', f"{safety_margin:.1f}%"]
    ]
    
    table = ax4.table(cellText=summary_data[1:], colLabels=summary_data[0],
                     cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    for i in range(len(summary_data)):
        for j in range(2):
            cell = table[(i, j)]
            if i == 0:
                cell.set_facecolor('#40466e')
                cell.set_text_props(color='white', weight='bold')
            else:
                cell.set_facecolor('#f1f1f2')
    
    ax4.set_title('估值总结', pad=20)
    
    # 5. 估值结论
    ax5 = plt.subplot(2, 3, 5)
    ax5.axis('off')
    
    intrinsic_value = valuation_result['每股价值(元)']
    safety_margin_val = (intrinsic_value - current_price) / current_price
    
    if safety_margin_val > 0.3:
        conclusion = "🟢 价值低估"
        recommendation = "强烈推荐买入"
    elif safety_margin_val > 0.1:
        conclusion = "🟡 轻度低估"
        recommendation = "可以考虑买入"
    elif safety_margin_val > -0.1:
        conclusion = "⚪ 合理估值"
        recommendation = "观望为主"
    else:
        conclusion = "🔴 价值高估"
        recommendation = "建议规避"
    
    conclusion_text = f"""
    **估值结论**
    
    {conclusion}
    
    • 内在价值: {intrinsic_value:.2f} 元
    • 安全边际: {safety_margin_val*100:.1f}%
    • 投资建议: {recommendation}
    
    **关键假设**
    • 折现率: {valuation_result['折现率']*100:.1f}%
    • 终期增长率: {valuation_result['终期增长率']*100:.1f}%
    • 历史增长率: {valuation_result['历史平均增长率']*100:.1f}%
    """
    
    ax5.text(0.1, 0.5, conclusion_text, fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 6. 多公司对比
    ax6 = plt.subplot(2, 3, 6)
    ax6.barh(valuation_df['公司名称'], valuation_df['每股价值(元)'], alpha=0.7)
    ax6.axvline(x=current_price, color='r', linestyle='--', label=f'当前价 {current_price:.1f}元')
    ax6.set_xlabel('每股价值 (元)')
    ax6.set_title('多公司估值对比')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 绘制第一个公司的 DCF 分析图
if valuation_results:
    plot_dcf_breakdown(valuation_results[0], current_price=25.0)

### 9.3 敏感性分析矩阵

In [ ]:
def sensitivity_analysis_matrix(cash_flow_df: pd.DataFrame, current_fcf: float,
                                discount_rates: list, terminal_rates: list) -> pd.DataFrame:
    """敏感性分析矩阵
    
    Args:
        cash_flow_df: 历史FCF数据
        current_fcf: 当前FCF
        discount_rates: 折现率列表
        terminal_rates: 终期增长率列表
    
    Returns:
        敏感性矩阵
    """
    matrix = []
    
    for dr in discount_rates:
        row = {'折现率': f'{dr*100:.1f}%'}
        for tr in terminal_rates:
            result = dcf_valuation(cash_flow_df, current_fcf, 
                                  discount_rate=dr, terminal_growth_rate=tr,
                                  net_debt=50.0, total_shares=10.0)
            row[f'{tr*100:.1f}%'] = result['每股价值(元)']
        matrix.append(row)
    
    df = pd.DataFrame(matrix)
    df.columns = ['折现率'] + [f'{tr*100:.1f}% 终期增长率' for tr in terminal_rates]
    return df

# 进行敏感性分析
discount_rates = [0.08, 0.09, 0.10, 0.11, 0.12]
terminal_rates = [0.02, 0.025, 0.03, 0.035, 0.04]

sensitivity_df = sensitivity_analysis_matrix(historical_data, current_fcf, discount_rates, terminal_rates)
print("DCF 估值敏感性分析矩阵 (每股价值，单位：元):")
sensitivity_df

In [ ]:
def plot_sensitivity_heatmap(sensitivity_df: pd.DataFrame, current_price: float = 25.0):
    """绘制敏感性分析热力图
    
    Args:
        sensitivity_df: 敏感性矩阵
        current_price: 当前股价
    """
    # 提取数值部分
    values = sensitivity_df.iloc[:, 1:].values
    
    # 计算安全边际
    safety_margins = (values - current_price) / current_price * 100
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. 每股价值热力图
    im1 = ax1.imshow(values, cmap='RdYlGn', aspect='auto', 
                     vmin=values.min(), vmax=values.max())
    
    # 设置坐标轴
    ax1.set_xticks(range(len(terminal_rates)))
    ax1.set_yticks(range(len(discount_rates)))
    ax1.set_xticklabels([f'{tr*100:.1f}%' for tr in terminal_rates])
    ax1.set_yticklabels([f'{dr*100:.1f}%' for dr in discount_rates])
    ax1.set_xlabel('终期增长率')
    ax1.set_ylabel('折现率')
    ax1.set_title('每股价值敏感性分析 (元)')
    
    # 添加数值标签
    for i in range(len(discount_rates)):
        for j in range(len(terminal_rates)):
            text = ax1.text(j, i, f'{values[i, j]:.1f}',
                          ha="center", va="center", color="black", fontsize=9)
    
    cbar1 = plt.colorbar(im1, ax=ax1)
    cbar1.set_label('每股价值 (元)')
    
    # 2. 安全边际热力图
    im2 = ax2.imshow(safety_margins, cmap='RdYlGn', aspect='auto',
                     vmin=-20, vmax=20)
    
    ax2.set_xticks(range(len(terminal_rates)))
    ax2.set_yticks(range(len(discount_rates)))
    ax2.set_xticklabels([f'{tr*100:.1f}%' for tr in terminal_rates])
    ax2.set_yticklabels([f'{dr*100:.1f}%' for dr in discount_rates])
    ax2.set_xlabel('终期增长率')
    ax2.set_ylabel('折现率')
    ax2.set_title('安全边际敏感性分析 (%)')
    
    # 添加数值标签
    for i in range(len(discount_rates)):
        for j in range(len(terminal_rates)):
            color = 'black' if -10 < safety_margins[i, j] < 10 else 'white'
            text = ax2.text(j, i, f'{safety_margins[i, j]:.1f}%',
                          ha="center", va="center", color=color, fontsize=9)
    
    cbar2 = plt.colorbar(im2, ax=ax2)
    cbar2.set_label('安全边际 (%)')
    
    plt.tight_layout()
    plt.show()

# 绘制敏感性热力图
plot_sensitivity_heatmap(sensitivity_df, current_price=25.0)

### 9.4 投资建议生成

In [ ]:
def generate_investment_recommendation(valuation_result: dict, current_price: float,
                                       sensitivity_df: pd.DataFrame) -> dict:
    """生成投资建议
    
    Args:
        valuation_result: DCF估值结果
        current_price: 当前股价
        sensitivity_df: 敏感性分析矩阵
    
    Returns:
        投资建议字典
    """
    intrinsic_value = valuation_result['每股价值(元)']
    safety_margin = (intrinsic_value - current_price) / current_price
    
    # 从敏感性矩阵中获取估值范围
    values = sensitivity_df.iloc[:, 1:].values
    min_value = values.min()
    max_value = values.max()
    
    # 投资评级
    if safety_margin > 0.3:
        rating = "强烈买入"
        risk_level = "低风险"
    elif safety_margin > 0.1:
        rating = "买入"
        risk_level = "中等风险"
    elif safety_margin > -0.1:
        rating = "持有"
        risk_level = "中等风险"
    else:
        rating = "卖出"
        risk_level = "高风险"
    
    return {
        '内在价值': f"{intrinsic_value:.2f} 元",
        '当前价格': f"{current_price:.2f} 元",
        '安全边际': f"{safety_margin*100:.1f}%",
        '估值范围': f"{min_value:.2f} - {max_value:.2f} 元",
        '投资评级': rating,
        '风险水平': risk_level,
        '买入点': f"{intrinsic_value * 0.8:.2f} 元" if safety_margin > 0 else "不建议买入",
        '目标价': f"{intrinsic_value * 1.2:.2f} 元" if safety_margin > 0 else "---",
        '止损点': f"{current_price * 0.85:.2f} 元" if safety_margin > 0 else "---"
    }

# 为所有公司生成投资建议
print("投资建议摘要：")
for result in valuation_results:
    recommendation = generate_investment_recommendation(result, 25.0, sensitivity_df)
    print(f"\n{result['公司名称']}：")
    for key, value in recommendation.items():
        print(f"  {key}: {value}")

## 10. 回测框架（概念验证）

**注意**：这是一个概念验证的简单回测框架，实际应用需要：
- 更完整的价格数据
- 交易成本模拟
- 风险管理
- 组合优化

In [ ]:
def simple_backtest(selected_companies: list, start_date: str, 
                    end_date: str, rebalance_freq: str = 'quarterly') -> dict:
    """简单回测框架
    
    Args:
        selected_companies: 筛选出的公司列表
        start_date: 回测开始日期
        end_date: 回测结束日期  
        rebalance_freq: 调仓频率
    
    Returns:
        回测结果
    """
    # 这里只是框架示例，实际需要：
    # 1. 获取历史价格数据
    # 2. 计算收益率
    # 3. 模拟调仓
    # 4. 计算组合收益
    
    print(f"回测框架概念演示：")
    print(f"  选定公司: {len(selected_companies)} 家")
    print(f"  回测期间: {start_date} ~ {end_date}")
    print(f"  调仓频率: {rebalance_freq}")
    
    # 模拟回测结果
    results = {
        'total_return': 0.18,  # 总收益 18%
        'annual_return': 0.09,  # 年化收益 9%
        'max_drawdown': -0.10,  # 最大回撤 -10%
        'sharpe_ratio': 1.3,   # 夏普比率 1.3
        'selected_stocks': selected_companies
    }
    
    print(f"\n模拟回测结果：")
    print(f"  总收益: {results['total_return']:.1%}")
    print(f"  年化收益: {results['annual_return']:.1%}")
    print(f"  最大回撤: {results['max_drawdown']:.1%}")
    print(f"  夏普比率: {results['sharpe_ratio']:.2f}")
    
    return results

# 模拟回测
selected_companies = [comp['name'] for comp in companies[:3]]  # 选择前3家公司
print("FCF 策略回测框架概念演示：")
backtest_results = simple_backtest(selected_companies, '2023-01-01', '2024-12-31')

## 11. 总结与实践建议

### 核心要点

1. **自由现金流是高质量财务指标**
   - 比利润更难造假
   - 反映企业真实造血能力

2. **FCF 因子有显著超额收益**
   - 高 FCF 公司通常有护城河
   - FCF 增长预示未来业绩

3. **多维度构建 FCF 因子**
   - 绝对值、相对比、增长率、稳定性
   - 不同行业有不同的合理范围

4. **DCF 估值需要谨慎**
   - 对输入参数敏感
   - 需要结合其他方法验证
   - 预测误差会影响估值结果

### 实践建议

1. **数据获取**：
   - 考虑使用专业数据源（如 Wind、同花顺）
   - 建立本地数据缓存

2. **因子优化**：
   - 结合行业特征调整标准
   - 加入市值、流动性约束

3. **风险控制**：
   - 分散投资，避免单一股票集中
   - 设置止损和仓位管理

4. **持续改进**：
   - 定期评估因子有效性
   - 结合其他因子构建多因子模型

### 下一步方向

1. **扩展数据集**：覆盖全部 A 股
2. **完善回测**：加入真实交易成本
3. **多因子组合**：结合动量、价值、质量因子
4. **机器学习**：用 ML 优化因子权重
5. **实盘验证**：小资金试验策略效果

### 本 notebook 特色

✅ **完全自包含** - 无需外部 API，使用模拟数据
✅ **最小依赖** - 仅需 pandas + numpy + matplotlib
✅ **立即可运行** - 完整流程，从数据到估值
✅ **教学友好** - 清晰的逻辑和可视化

---

## 附录：常见问题

### Q1: 为什么使用模拟数据而不是真实数据？
**A**: 本 notebook 专注于教学和概念验证，使用模拟数据可以：
- 避免外部 API 依赖
- 立即运行完整流程
- 便于理解估值原理
- 可控参数便于测试

### Q2: 如何将此框架应用到真实数据？
**A**: 
- 替换 `create_mock_company_financials()` 为真实数据获取
- 处理真实数据中的缺失值和异常值
- 考虑行业特征调整参数
- 加入更多风险因子

### Q3: FCF 因子适合所有行业吗？
**A**: 不适合。银行业现金流表结构特殊，高增长行业 FCF 可能为负但前景好。

### Q4: 如何判断 FCF 因子的有效性？
**A**: 
- 历史回测分析 IC、IR 值
- 分层回测看超额收益
- 行业中性化处理

### Q5: DCF 估值的局限性是什么？
**A**:
- 对输入参数高度敏感（折现率、增长率）
- 预测未来现金流的不确定性
- 终值占比过高，易失真
- 不适合周期性行业

**核心原则**: FCF 和 DCF 都是重要的分析工具，但不应作为唯一决策依据，需要结合其他分析方法和市场环境综合判断。